# Phase 0 기술 검증 PoC

목표: 현재 스택이 실제로 동작 가능한지 노트북에서 검증한다.

검증 범위:
- Ollama + Gemma API, tool calling, 응답 시간
- PydanticAI tool calling, structured output, tool retry
- Open Interpreter 설치/임포트/제어 가능성
- Permission hook: auto_run=False 스타일 승인 대기, 코드 추출, 실행 직전 인터셉트, 승인 후 실행 재개

In [4]:
import asyncio
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
import textwrap
import time
from dataclasses import dataclass, field
from pathlib import Path

import httpx
from pydantic import BaseModel, Field

ROOT = Path.cwd()
MODEL = "gemma4:latest"
OLLAMA_URL = "http://127.0.0.1:11434"

results = {}

def record(name, ok, detail):
    results[name] = {"ok": bool(ok), "detail": detail}
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}")
    if isinstance(detail, (dict, list)):
        print(json.dumps(detail, ensure_ascii=False, indent=2, default=str))
    else:
        print(detail)

## 1. Ollama + Gemma

Gemma 모델은 원문 계획에서 취소선 처리되어 있지만, 완료 조건에 `Gemma -> Tool 호출 성공`이 있어 최소 검증을 실행한다.

In [5]:
try:
    t0 = time.perf_counter()
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()
    elapsed = round(time.perf_counter() - t0, 3)
    models = [m["name"] for m in tags.get("models", [])]
    gemma_meta = next((m for m in tags.get("models", []) if m.get("name") == MODEL), None)
    record(
        "Ollama API / Gemma model",
        gemma_meta is not None,
        {
            "elapsed_sec": elapsed,
            "models": models,
            "gemma_capabilities": gemma_meta.get("capabilities") if gemma_meta else None,
        },
    )
except Exception as exc:
    record("Ollama API / Gemma model", False, repr(exc))

[PASS] Ollama API / Gemma model
{
  "elapsed_sec": 0.171,
  "models": [
    "gemma4:latest",
    "deepseek-r1:14b"
  ],
  "gemma_capabilities": [
    "completion",
    "tools",
    "thinking"
  ]
}


In [6]:
try:
    import ollama

    tool_calls = []

    def add_numbers(a: int, b: int) -> int:
        return a + b

    tools = [
        {
            "type": "function",
            "function": {
                "name": "add_numbers",
                "description": "Add two integers.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "integer"},
                        "b": {"type": "integer"},
                    },
                    "required": ["a", "b"],
                },
            },
        }
    ]

    t0 = time.perf_counter()
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "Call the add_numbers tool with a=17 and b=25. Do not solve mentally.",
            }
        ],
        tools=tools,
    )
    elapsed = round(time.perf_counter() - t0, 3)
    message = response.get("message", {})
    tool_calls = message.get("tool_calls") or []
    first_call = tool_calls[0] if tool_calls else None
    fn = first_call.get("function", {}) if first_call else {}
    args = fn.get("arguments", {}) if fn else {}
    tool_result = add_numbers(**args) if fn.get("name") == "add_numbers" else None
    record(
        "Gemma tool calling via Ollama",
        fn.get("name") == "add_numbers" and tool_result == 42,
        {
            "elapsed_sec": elapsed,
            "tool_name": fn.get("name"),
            "arguments": args,
            "tool_result": tool_result,
            "raw_message": message.model_dump() if hasattr(message, "model_dump") else repr(message),
        },
    )
except Exception as exc:
    record("Gemma tool calling via Ollama", False, traceback.format_exc())

NameError: name 'traceback' is not defined

## 2. PydanticAI

SDK 기능 검증은 deterministic `TestModel`로 실행한다. 로컬 LLM 품질 변동과 분리해서 PydanticAI 자체의 tool calling, structured output, tool retry가 되는지를 확인한다.

In [ ]:
try:
    from pydantic_ai import Agent, ModelRetry
    from pydantic_ai.models.test import TestModel

    class CalcResult(BaseModel):
        answer: int = Field(description="Final arithmetic answer")
        tool_used: bool

    retry_state = {"calls": 0}
    agent = Agent(TestModel(), output_type=CalcResult)

    @agent.tool_plain(retries=1)
    def flaky_add(a: int, b: int) -> int:
        retry_state["calls"] += 1
        if retry_state["calls"] == 1:
            raise ModelRetry("Temporary tool failure; retry once.")
        return a + b

    async def run_pydanticai_test():
        return await agent.run("Use flaky_add to add 2 and 5, then return structured output.")

    result = asyncio.run(run_pydanticai_test())
    output = result.output
    record(
        "PydanticAI tool calling / structured output / tool retry",
        isinstance(output, CalcResult) and retry_state["calls"] == 2,
        {
            "output_type": type(output).__name__,
            "output": output.model_dump(),
            "tool_call_count": retry_state["calls"],
        },
    )
except Exception:
    record(
        "PydanticAI tool calling / structured output / tool retry",
        False,
        traceback.format_exc(),
    )

In [ ]:
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.ollama import OllamaModel
    from pydantic_ai.providers.ollama import OllamaProvider

    class ShortAction(BaseModel):
        action: str
        content: str

    agent = Agent(
        OllamaModel("gemma4", provider=OllamaProvider(base_url=f"{OLLAMA_URL}/v1")),
        output_type=ShortAction,
        system_prompt="Return only structured data. action must be 'write'.",
    )

    async def run_gemma_structured():
        return await agent.run("Create a short write action with content 'poc-ok'.")

    t0 = time.perf_counter()
    result = asyncio.run(run_gemma_structured())
    elapsed = round(time.perf_counter() - t0, 3)
    record(
        "PydanticAI + Gemma structured output",
        isinstance(result.output, ShortAction) and result.output.action == "write",
        {"elapsed_sec": elapsed, "output": result.output.model_dump()},
    )
except Exception:
    record("PydanticAI + Gemma structured output", False, traceback.format_exc())

## 3. Open Interpreter

현재 `.venv`는 Python 3.14이며, `open-interpreter` 배포판이 이 환경에서 설치/임포트 가능한지 확인한다.

In [ ]:
try:
    has_interpreter = importlib.util.find_spec("interpreter") is not None
    if has_interpreter:
        from interpreter import interpreter

        interpreter.auto_run = False
        record(
            "Open Interpreter import / auto_run control",
            True,
            {
                "module": "interpreter",
                "auto_run": interpreter.auto_run,
            },
        )
    else:
        pip = subprocess.run(
            [sys.executable, "-m", "pip", "install", "open-interpreter"],
            text=True,
            capture_output=True,
            timeout=120,
        )
        record(
            "Open Interpreter import / install",
            False,
            {
                "python": sys.version,
                "importable": False,
                "pip_returncode": pip.returncode,
                "pip_tail": (pip.stdout + pip.stderr)[-2000:],
            },
        )
except Exception:
    record("Open Interpreter import / install", False, traceback.format_exc())

## 4. Permission Hook 검증

Open Interpreter가 현재 환경에서 실행되지 않아도, 필요한 제어 모델을 최소 구현으로 검증한다.

검증 항목:
- `auto_run=False` 상태에서 승인 전 실행 대기
- 생성 코드 추출
- 실행 직전 hook/intercept
- 승인 후 실행 재개

In [ ]:
@dataclass
class PermissionGate:
    auto_run: bool = False
    intercepted: list = field(default_factory=list)

    def extract_code(self, message: str) -> str:
        marker = "```python"
        start = message.index(marker) + len(marker)
        end = message.index("```", start)
        return message[start:end].strip()

    def before_execute(self, code: str):
        event = {"code": code, "approved": self.auto_run}
        self.intercepted.append(event)
        if not self.auto_run:
            return {"status": "waiting_for_approval", "code": code}
        return self.execute(code)

    def approve_and_resume(self, code: str):
        self.auto_run = True
        return self.before_execute(code)

    def execute(self, code: str):
        namespace = {"ROOT": ROOT}
        exec(code, namespace, namespace)
        return {"status": "executed", "created": str(namespace.get("created"))}

generated_message = '''
파일을 만들기 위해 다음 Python 코드를 실행합니다.

```python
created = ROOT / "permission_hook_created.txt"
created.write_text("permission hook approved", encoding="utf-8")
```
'''

created_path = ROOT / "permission_hook_created.txt"
if created_path.exists():
    created_path.unlink()

gate = PermissionGate(auto_run=False)
extracted = gate.extract_code(generated_message)
pending = gate.before_execute(extracted)
exists_before = created_path.exists()
resumed = gate.approve_and_resume(extracted)
exists_after = created_path.exists()
content_after = created_path.read_text(encoding="utf-8") if exists_after else None

record(
    "Permission hook approval flow",
    pending["status"] == "waiting_for_approval"
    and exists_before is False
    and resumed["status"] == "executed"
    and exists_after
    and content_after == "permission hook approved",
    {
        "auto_run_initial": False,
        "extracted_code": extracted,
        "pending": pending,
        "exists_before_approval": exists_before,
        "intercept_count": len(gate.intercepted),
        "resumed": resumed,
        "exists_after_approval": exists_after,
        "content_after": content_after,
    },
)

## 최종 판정

In [ ]:
print(json.dumps(results, ensure_ascii=False, indent=2))

required = [
    "Ollama API / Gemma model",
    "Gemma tool calling via Ollama",
    "PydanticAI tool calling / structured output / tool retry",
    "Permission hook approval flow",
]
open_interpreter_ok = any(
    name.startswith("Open Interpreter") and value.get("ok")
    for name, value in results.items()
)
all_required_ok = all(results.get(name, {}).get("ok") for name in required) and open_interpreter_ok

print()
print("완료 조건 판정")
print(f"- Gemma -> Tool 호출 성공: {results.get('Gemma tool calling via Ollama', {}).get('ok')}")
print(f"- Open Interpreter 제어 성공: {open_interpreter_ok}")
print(f"- 실행 전 승인 대기 구현 가능: {results.get('Permission hook approval flow', {}).get('ok')}")
print(f"- 핵심 필수 항목 전체: {all_required_ok}")